<a href="https://colab.research.google.com/github/Juliusp36/GinyuForce2/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Part A: Load the MovieLens ratings

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists('ml-100k'):
  urllib.request.urlretrieve(
      'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
      'ml-100k.zip')
  zipfile.ZipFile('ml-100k.zip').extractall()

ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=['user', 'item', 'rating', 'time'])

print(ratings.head())
print('Number of ratings:', len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


The code above downloads the movielens 100k dataset from the internet and loads the rating into a table with the columns of users, items, rating, time.

2 Part B: Make an honest train/test split

In [ ]:
def split_ratings(ratings, test_frac=0.10, seed=0):

  rng = np.random.default_rng(seed)
  is_test = np.zeros(len(ratings), dtype=bool)

  user_left = ratings.groupby('user').size().to_dict()
  item_left = ratings.groupby('item').size().to_dict()

  for pos in rng.permutation(len(ratings)):
    row = ratings.iloc[pos]
    user, item = row['user'], row['item']

    if (rng.random() < test_frac
        and user_left[user] > 1
        and item_left[item] > 1):
      is_test[pos] = True
    user_left[user] -= 1
    item_left[item] -= 1
  return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print('train:', len(train), 'test:', len(test))
print('test fraction:', round(len(test) / len(ratings), 3))


train: 90246 test: 9754
test fraction: 0.098


The code splits the ratings into train and set. It vists every rating in random order. For each one, it moves the trating to the test set only if it meet all the statements in the if statement.

3. Part C: Build the training matrix

In [ ]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
  R = np.full((N_USERS, N_ITEMS), np.nan)
  R[df['user'].to_numpy() -1,
    df['item'].to_numpy() -1] = df['rating'].to_numpy()
  return R

R = to_matrix(train)
known = np.isnan(R)

print('Matrix shape:', R.shape)
print('Known training rating:', known.sum())


Matrix shape: (943, 1682)
Known training rating: 1495880


The code turns the training rating from a list of two into a user-by-movie matrix. Which is where you get the matrix shape: (943, 1682) from.

In [ ]:
truth = test['rating'].to_numpy()
u = test['user'].to_numpy() - 1
i = test['item'].to_numpy() - 1

This stores the real number of start for the rating of test. U is the user while i is the item.

4 Part D: Score predictions with RMSE

In [ ]:
def rmse(pred, truth):
  return np.sqrt(np.mean((pred - truth)**2))

The code squares the mean of pred-truth then takes the square root.

5 Part E: Build simple baselines

In [ ]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0) [has_rating] / n_item[has_rating])

print('Overall average:', round(mu, 3))

Overall average: 3.53


The code compute the average ratings used as baseline predictors. One overall, one per user, and one per moives.

In [ ]:
scores = {}
scores['always 3'] = rmse(np.full(len(test), 3.0), truth)
scores['overall average'] = rmse(np.full(len(test), mu), truth)
scores['user average'] = rmse(user_mean[u], truth)
scores['movie average'] = rmse(item_mean[i], truth)
for name, score in scores.items():
  print(f'{name:16s} {score:.4f}')

always 3         1.2514
overall average  1.1322
user average     1.0414
movie average    2.9720


The code scores four baseline predictors on the test set and prints them out. You have "always 3", "overall average", "user average", "movie average". Always 3 predit 3.0 for every test rating. Overall average predicts mu. User average look up each test rating;s user and predicts that user's own average. Movie average looks up each test rating's movie and predicts the movie's average.

6 Part F: Combine user and movie effect

In [ ]:
def bias_baseline(R):

  known = ~np.isnan(R)
  mu = np.nanmean(R)

  b_user = np.nanmean(R - mu, axis=1)
  residual = R - mu - b_user[:, None]

  n_item = known.sum(axis=0)
  b_item = np.zeros(R.shape[1])
  has_rating = n_item > 0
  b_item[has_rating] = (
      np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])
  return mu, b_user, b_item

The code combines the overall averages, adjustments from the users, adjustments from the movie and that becomes the bias.

In [ ]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

scores['user + movie effects'] = rmse(pred, truth)
print(pd.Series(scores, name='test RMSE').round(4).sort_values())

user + movie effects    0.9536
user average            1.0414
overall average         1.1322
always 3                1.2514
movie average           2.9720
Name: test RMSE, dtype: float64


The code uses the biases to make predictions on the test set, scores them, and prints them. For each test rating, it looks up that rating's user and movie and adds up the overall average + user bias + moive bias.
  _

Practice Exercises

Exercise 1

In [49]:
R_tiny = np.array([[5, 3, np.nan],
                   [4, np.nan, 1],
                   [np.nan, 2, 2]])
mu_t, b_user_t, b_item_t = bias_baseline(R_tiny)
print(mu_t, b_user_t, b_item_t)

2.8333333333333335 [ 1.16666667 -0.33333333 -0.83333333] [ 1.25 -0.5  -0.75]


The code create a 3-user by 3-moive rating matrix. np.nan marks a rating the user hasn't given.

Exercise 2

In [54]:
R_all = to_matrix(ratings)
cheat_user_mean = np.nanmean(R_all, axis=1)
cheat_score = rmse(cheat_user_mean[u], truth)
print('cheating:', round(cheat_score, 4), ' honest:', round(scores['user average'], 4))

cheating: 1.0311  honest: 1.0414


The RMSE is lower than the honest user. It is misleading because the user average were computed using the very rating they're being tested on. The test score no longer measures how well the model handles rating it hasn't seen. which is the whole point of a test set. The code measures how much a model improves if it's allowed to peek at the test.

In [56]:
def run_all(seed):
    train, test = split_ratings(ratings, seed=seed)
    R = to_matrix(train)
    known = ~np.isnan(R)
    u = test['user'].to_numpy() - 1
    i = test['item'].to_numpy() - 1
    truth = test['rating'].to_numpy()

    mu = np.nanmean(R)
    user_mean = np.nanmean(R, axis=1)
    n_item = known.sum(axis=0)
    item_mean = np.full(N_ITEMS, mu)
    has = n_item > 0
    item_mean[has] = np.nansum(R, axis=0)[has] / n_item[has]

    mu2, b_user, b_item = bias_baseline(R)
    pred = np.clip(mu2 + b_user[u] + b_item[i], 1, 5)

    return {'always 3': rmse(np.full(len(test), 3.0), truth),
            'overall average': rmse(np.full(len(test), mu), truth),
            'user average': rmse(user_mean[u], truth),
            'movie average': rmse(item_mean[i], truth),
            'user + movie effects': rmse(pred, truth)}

print(pd.DataFrame({'seed 0': run_all(0), 'seed 1': run_all(1)}).round(4))

                      seed 0  seed 1
always 3              1.2514  1.2337
overall average       1.1322  1.1100
user average          1.0414  1.0336
movie average         1.0247  1.0138
user + movie effects  0.9536  0.9565


The values do change slightly. The ranking stays the same for the model. This would tell me a single test RMSE depend partly on which rating it landed in the test set. This also mean that a difference between two recommenders smaller than the split to split could just be luck. The code reruns the entire pipeline for a given random seed, then put the results for seed o and seed 1 side by side. This way you can see whether the numbers and the ranking change.